In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# Setup
df = pd.read_csv('../data/cleaned/df_main.csv')
os.makedirs('../reports', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2D6A4F','#40916C','#52B788','#74C69D','#95D5B2',
          '#B7E4C7','#D8F3DC','#1B4332','#081C15','#52B788']
print(f"Loaded df_main: {df.shape}")
print("Setup complete")

In [ ]:
# CHART 1: Delivery Timing vs Customer Satisfaction
# This is your HEADLINE chart — most important finding

def bucket_delay(d):
    if pd.isna(d): return None
    if d > 0: return '3. Late'
    elif d >= -7: return '2. Slightly Early\n(0-7 days)'
    elif d >= -14: return '1. Early\n(8-14 days)'
    else: return '0. Very Early\n(15+ days)'

df['delivery_category'] = df['delivery_delay_days'].apply(bucket_delay)
chart1 = df[df['review_score'].notna()].groupby('delivery_category').agg(
    avg_review=('review_score','mean'),
    total_orders=('order_id','nunique')
).reset_index().sort_values('delivery_category')

fig, ax1 = plt.subplots(figsize=(10,6))
colors = ['#52B788','#40916C','#2D6A4F','#E24B4A']
bars = ax1.bar(chart1['delivery_category'],
               chart1['avg_review'], color=colors, width=0.5,
               edgecolor='white', linewidth=1.5)
ax1.set_ylim(0, 5.5)
ax1.set_ylabel('Average Review Score (★)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Delivery Timing vs Estimate', fontsize=12, fontweight='bold')
ax1.set_title('Late Delivery Cuts Customer Satisfaction Nearly in Half\n'
              'Olist Platform — 96,478 Delivered Orders',
              fontsize=14, fontweight='bold', pad=15)

for bar, (_, row) in zip(bars, chart1.iterrows()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{row["avg_review"]:.2f}★\n({row["total_orders"]:,} orders)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

ax1.axhline(y=4.0, color='orange', linestyle='--',
            alpha=0.7, label='4.0★ threshold')
ax1.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/chart1_delivery_vs_satisfaction.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart 1 saved")

In [ ]:
# CHART 2: Revenue by Product Category (Top 10)
cat_rev = df.groupby('product_category_name_english').agg(
    total_revenue=('price','sum'),
    total_orders=('order_id','nunique')
).reset_index()
cat_rev = cat_rev[cat_rev['product_category_name_english'] != 'unknown']
cat_rev = cat_rev.nlargest(10, 'total_revenue')
cat_rev['avg_order_value'] = cat_rev['total_revenue'] / cat_rev['total_orders']
cat_rev = cat_rev.sort_values('total_revenue')

fig, ax = plt.subplots(figsize=(10,7))
bars = ax.barh(cat_rev['product_category_name_english'],
               cat_rev['total_revenue'] / 1e6,
               color='#40916C', edgecolor='white', linewidth=1)

for bar, (_, row) in zip(bars, cat_rev.iterrows()):
    ax.text(bar.get_width() + 0.01,
            bar.get_y() + bar.get_height()/2,
            f'₹{row["total_revenue"]/1e6:.2f}M\n'
            f'({row["total_orders"]:,} orders | '
            f'avg ₹{row["avg_order_value"]:.0f})',
            va='center', fontsize=8.5)

ax.set_xlabel('Total Revenue (₹ Millions)', fontsize=11, fontweight='bold')
ax.set_title('Top 10 Product Categories by Revenue\n'
             'Health & Beauty leads in revenue; Watches/Gifts most efficient per order',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, cat_rev['total_revenue'].max()/1e6 * 1.35)
plt.tight_layout()
plt.savefig('../reports/chart2_revenue_by_category.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart 2 saved")

In [ ]:
# CHART 3: Monthly Revenue Trend with Growth Rate
df['order_purchase_timestamp'] = pd.to_datetime(
    df['order_purchase_timestamp'], errors='coerce')
df['order_month_dt'] = df['order_purchase_timestamp'].dt.to_period('M')

monthly = df.groupby('order_month_dt').agg(
    revenue=('price','sum'),
    orders=('order_id','nunique')
).reset_index()
monthly['order_month_dt'] = monthly['order_month_dt'].astype(str)
# Remove incomplete months (first and last)
monthly = monthly.iloc[1:-1].reset_index(drop=True)
monthly['mom_growth'] = monthly['revenue'].pct_change() * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12,8), sharex=True)

# Revenue line
ax1.fill_between(range(len(monthly)), monthly['revenue']/1e6,
                 alpha=0.3, color='#40916C')
ax1.plot(range(len(monthly)), monthly['revenue']/1e6,
         color='#2D6A4F', linewidth=2.5, marker='o', markersize=4)
ax1.set_ylabel('Revenue (₹ Millions)', fontsize=10, fontweight='bold')
ax1.set_title('Olist Monthly Revenue Trend — Strong Growth with Nov 2017 Peak\n'
              '(Jan 2017 – Aug 2018)', fontsize=13, fontweight='bold')

# Growth rate bars
colors_growth = ['#E24B4A' if x < 0 else '#40916C'
                 for x in monthly['mom_growth'].fillna(0)]
ax2.bar(range(len(monthly)), monthly['mom_growth'].fillna(0),
        color=colors_growth, alpha=0.8, edgecolor='white')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_ylabel('MoM Growth %', fontsize=10, fontweight='bold')

# X-axis labels
step = max(1, len(monthly)//10)
ax2.set_xticks(range(0, len(monthly), step))
ax2.set_xticklabels(monthly['order_month_dt'].iloc[::step],
                    rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/chart3_monthly_revenue_trend.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart 3 saved")

In [ ]:
# CHART 4: Customer Segments (RFM Preview)
# Quick state-level heatmap as a bonus visual
state_metrics = df.groupby('customer_state').agg(
    total_orders=('order_id','nunique'),
    avg_review=('review_score','mean'),
    avg_delivery_delay=('delivery_delay_days','mean')
).reset_index()
state_metrics = state_metrics[state_metrics['total_orders'] > 100]

fig, ax = plt.subplots(figsize=(10,6))
scatter = ax.scatter(state_metrics['avg_delivery_delay'],
                     state_metrics['avg_review'],
                     s=state_metrics['total_orders']/10,
                     c=state_metrics['total_orders'],
                     cmap='Greens', alpha=0.8, edgecolors='#2D6A4F', linewidth=1)

for _, row in state_metrics.iterrows():
    ax.annotate(row['customer_state'],
                (row['avg_delivery_delay'], row['avg_review']),
                fontsize=8, ha='center', va='bottom',
                xytext=(0,5), textcoords='offset points')

plt.colorbar(scatter, label='Total Orders')
ax.set_xlabel('Avg Delivery Delay (days, negative = early)',
              fontsize=11, fontweight='bold')
ax.set_ylabel('Avg Review Score (★)', fontsize=11, fontweight='bold')
ax.set_title('State Performance: Delivery vs Satisfaction\n'
             '(Bubble size = order volume)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/chart4_state_performance.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart 4 saved")